# Data Download

This notebook downloads the requisite data needed to draft fantasy baseball players. It creates a data directory and stores the data that is easily and quickly digestable into PANDAS DataFrames

## Import libraries, configure seasons, and enable PyBaseball cache


In [1]:
from __future__ import annotations

from datetime import date
from typing import Dict, List

import pandas as pd
from pybaseball import (
    batting_stats,
    cache,
    pitching_stats,
    standings,
    team_batting,
    team_pitching,
)

# Rely on pybaseball's built-in cache (version-safe API call).
cache.enable()

CURRENT_YEAR = date.today().year
YEARS_TO_PULL = [CURRENT_YEAR - 3, CURRENT_YEAR - 2, CURRENT_YEAR - 1]

# Set to True to include additional, potentially slower example pulls.
RUN_OPTIONAL_EXAMPLES = False


## Helper functions (PyBaseball primer utilities)


In [2]:
def normalize_years(years: List[int]) -> List[int]:
    """Return unique, sorted, valid MLB years."""
    valid_years = sorted({year for year in years if 1871 <= year <= CURRENT_YEAR})
    if not valid_years:
        raise ValueError('No valid years provided. Update YEARS_TO_PULL with MLB seasons.')
    return valid_years


def fetch_season_frames(year: int) -> Dict[str, pd.DataFrame]:
    """Download season-level hitter/pitcher leaderboards and attach season labels."""
    hitters_df = batting_stats(year, qual=0)
    pitchers_df = pitching_stats(year, qual=0)

    hitters_df = hitters_df.copy()
    pitchers_df = pitchers_df.copy()

    hitters_df['Season'] = year
    pitchers_df['Season'] = year

    return {'batting': hitters_df, 'pitching': pitchers_df}


def dataset_health(df: pd.DataFrame, dataset_name: str) -> Dict[str, object]:
    """Quick diagnostics to confirm data was downloaded and is queryable."""
    return {
        'dataset': dataset_name,
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1]),
        'null_cells': int(df.isna().sum().sum()),
        'sample_columns': ', '.join(df.columns[:8]),
    }


## Download three seasons of draft-relevant data with PyBaseball


In [3]:
years = normalize_years(YEARS_TO_PULL)

season_data: Dict[int, Dict[str, pd.DataFrame]] = {}
all_batting: List[pd.DataFrame] = []
all_pitching: List[pd.DataFrame] = []

for year in years:
    print(f'Downloading pybaseball leaderboards for {year}...')
    result = fetch_season_frames(year)
    season_data[year] = result

    all_batting.append(result['batting'])
    all_pitching.append(result['pitching'])

batting_all_years = pd.concat(all_batting, ignore_index=True)
pitching_all_years = pd.concat(all_pitching, ignore_index=True)

print('Download complete.')
print(f'Combined batting rows: {len(batting_all_years):,}')
print(f'Combined pitching rows: {len(pitching_all_years):,}')


## Integrity checks: verify downloaded data can be queried


In [4]:
health_report = pd.DataFrame(
    [
        dataset_health(batting_all_years, 'batting_all_years'),
        dataset_health(pitching_all_years, 'pitching_all_years'),
    ]
)

health_report


## Primer examples: common PyBaseball workflows


In [5]:
# 1) Top fantasy-relevant hitters by HR and SB from the most recent pulled season.
most_recent_year = years[-1]
recent_batting = season_data[most_recent_year]['batting']

recent_batting[['Name', 'Team', 'HR', 'SB', 'R', 'RBI', 'AVG']].sort_values(
    by=['HR', 'SB'], ascending=False
).head(15)

# 2) Top fantasy-relevant pitchers by strikeouts and saves from the most recent year.
recent_pitching = season_data[most_recent_year]['pitching']

recent_pitching[['Name', 'Team', 'W', 'SV', 'SO', 'ERA', 'WHIP']].sort_values(
    by=['SO', 'SV'], ascending=False
).head(15)

# 3) Team-level snapshots often useful for draft context.
team_batting_recent = team_batting(most_recent_year)
team_pitching_recent = team_pitching(most_recent_year)

# Show small previews to verify these endpoints are usable.
display(team_batting_recent.head(10))
display(team_pitching_recent.head(10))

# 4) League standings are a useful contextual feature.
league_tables = standings(most_recent_year)
if league_tables:
    display(league_tables[0].head())

# 5) Optional examples: keep off by default to avoid unnecessary calls.
if RUN_OPTIONAL_EXAMPLES:
    print('Optional examples enabled: add statcast/playerid lookups here as needed.')


In [ ]:
# End of primer notebook section.
